# 编写和运行任务
学习编写任务的基础知识。

在 Prefect 工作流中，完美的任务是独立的工作单元。你可以通过给任何 Python 函数添加 `@task` 装饰器来将其转变为任务。任务可以：

- 接受输入，执行工作，并返回输出
- 跨调用缓存它们的执行结果
- 将工作流逻辑封装成可重用单元，跨越不同流程
- 在运行前接收关于上游任务依赖及其状态的元数据
- 使用自动[日志记录](https://docs.prefect.io/v3/develop/logging)捕获运行时详情、标签和最终状态
- 并发执行
- 在同一文件中定义或从模块中导入
- 可以从工作流或其他任务中调用

工作流和任务共享一些通用特性：

- 它们可以使用各自的装饰器进行定义，该装饰器接受配置设置（见所有[任务设置](https://docs.prefect.io/v3/develop/write-tasks#task-configuration)和[工作流设置](https://docs.prefect.io/v3/develop/write-flows#flow-settings)）
- 它们可以有一个名称、描述和标签，用于组织和记录
- 它们提供重试、超时和其他钩子来处理失败和完成事件

这里有包含单一任务的简单流示例：

```python
import httpx
from prefect import flow, task
from typing import Optional


@task
def get_url(url: str, params: Optional[dict[str, any]] = None):
    response = httpx.get(url, params=params)
    response.raise_for_status()
    return response.json()


@flow(retries=3, retry_delay_seconds=5, log_prints=True)
def get_repo_info(repo_name: str = "PrefectHQ/prefect"):
    url = f"https://api.github.com/repos/{repo_name}"
    repo_stats = get_url(url)
    print(f"{repo_name} repository statistics 🤓:")
    print(f"Stars 🌠 : {repo_stats['stargazers_count']}")
    print(f"Forks 🍴 : {repo_stats['forks_count']}")


if __name__ == "__main__":
    get_repo_info()
```

运行结果：
```bash
09:55:55.412 | INFO    | prefect.engine - Created flow run 'great-ammonite' for flow 'get-repo-info'
09:55:55.499 | INFO    | Flow run 'great-ammonite' - Created task run 'get_url-0' for task 'get_url'
09:55:55.500 | INFO    | Flow run 'great-ammonite' - Executing 'get_url-0' immediately...
09:55:55.825 | INFO    | Task run 'get_url-0' - Finished in state Completed()
09:55:55.827 | INFO    | Flow run 'great-ammonite' - PrefectHQ/prefect repository statistics 🤓:
09:55:55.827 | INFO    | Flow run 'great-ammonite' - Stars 🌠 : 12157
09:55:55.827 | INFO    | Flow run 'great-ammonite' - Forks 🍴 : 1251
09:55:55.849 | INFO    | Flow run 'great-ammonite' - Finished in state Completed('All states completed.')
```

此任务运行也会在用户界面中进行追踪。

任务通过任务键来唯一识别，该任务键是由任务名称、函数的完全限定名以及任何标签组成的哈希值。如果任务没有指定名称，那么名称将从装饰过的函数对象中派生出来。

```{admonition} 任务应该有多大？
:class: tip

Prefect 推荐使用“小任务”。每个任务都应该代表工作流中的单一逻辑步骤。这样可以更好地控制任务失败的影响。

虽然你可以将所有代码放在一个任务中，但任何一行代码的失败都会导致整个任务失败，并且必须从头开始重试。通过将代码分割成多个相互依赖的任务，可以避免这种情况。
```

## 支持的函数

几乎任何标准的 Python 函数都可以通过添加 `@task` 装饰器转变为 Prefect 任务。

Prefect 默认使用客户端的任务运行编排，这显著提高了性能，尤其是对于包含许多任务的工作流。任务创建和状态更新在本地进行，减少了执行期间对 Prefect 服务器的 API 调用。这使得能够高效处理大规模工作流，并在服务器连接不稳定时提高可靠性。